# 1. Reshaping Training and Test Data

In [6]:
import numpy as np

X_train = np.load("/Users/mayawiegand/Documents/ECS 171/Project/music-genre-classification/processed_data/X_train.npy")
X_test = np.load("/Users/mayawiegand/Documents/ECS 171/Project/music-genre-classification/processed_data/X_test.npy")
y_train = np.load("/Users/mayawiegand/Documents/ECS 171/Project/music-genre-classification/processed_data/y_train.npy")
y_test = np.load("/Users/mayawiegand/Documents/ECS 171/Project/music-genre-classification/processed_data/y_test.npy")

print(X_train.shape) # spectrogram is samples, number of mel filters (n_mels), number of time frames
# need to flatten this as KNN can't take 3D data

X_train = X_train.reshape(X_train.shape[0], -1) # this transforms the data to number of samples, n_mels * n time frames
X_test = X_test.reshape(X_test.shape[0], -1) 

print(X_train.shape)


(7987, 128, 130)
(7987, 16640)


# 2. Label Encoding
- Needed since some scikit-learn functions require numeric-based class labels rather than strings/categorical
- Converts categorical class labels to integers so they can be processed correctly
    - Not the same as encoding features, as this doesn't influence model predictions

In [7]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train_enc = le.fit_transform(y_train) # fitting on the training, and applying this transformation
y_test_enc  = le.transform(y_test) # applying transformation only (using the training fit)

# 3. Fitting Baseline KNN

In [ ]:
# can take this out in final code, just want to see what this looks like

from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=50) # initializing KNN model with k = 50

knn.fit(X_train, y_train_enc) # fitting model on training data

y_pred_enc = knn.predict(X_test) # forming predictions by applying fitted model

y_pred_labels = le.inverse_transform(y_pred_enc) # converting the encoded labels back to the genre names
print(y_pred_labels)

['classical' 'classical' 'classical' ... 'metal' 'metal' 'classical']


In [ ]:
# quick analysis measure

from sklearn.metrics import f1_score
f1 = f1_score(y_test, y_pred_labels, average='macro') # using labels so true and predicted values match in datatype
f1 # super low, tuning hyperparameters should help with this

0.32660185618341375

# 4. Hyperparameter Tuning Setup

In [ ]:
from skopt import BayesSearchCV
from skopt.space import Integer, Categorical

knn = KNeighborsClassifier() # reinitializes this to prevent carying over from the base model fit

# defining the grid of values we want to test
search_space = {
    "n_neighbors": Integer(1,80),
    "weights": Categorical(["uniform", "distance"]),
    "metric": Categorical(["manhattan", "euclidean", "cosine"])
}

bayes_search = BayesSearchCV(estimator = knn,
                           search_spaces=search_space,
                           scoring = "f1_macro",
                           n_iter=50, # the number of hyperparameter combinations to try
                           cv = 5,
                           n_jobs=-1, # using all CPU cores to speed up search
                           random_state=42,
                           verbose=1 # prints progress messages during tuning process
)

bayes_search.fit(X_train, y_train_enc)

Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits


,estimator,KNeighborsClassifier()
,search_spaces,"{'metric': Categorical(c...), prior=None), 'n_neighbors': Integer(low=1...m='normalize'), 'weights': Categorical(c...), prior=None)}"
,optimizer_kwargs,None
,n_iter,2
,scoring,'f1_macro'
,fit_params,None
,n_jobs,-1
,n_points,1
,iid,'deprecated'
,refit,True
,cv,5


In [11]:
bayes_search.best_params_

OrderedDict([('metric', 'manhattan'),
             ('n_neighbors', 58),
             ('weights', 'distance')])

# 6. PCA
- Will also perform PCA, a dimension-reduction technique and compare model performance in both non-PCA predictions and PCA predictions

In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components = 0.95) # defining PCA using the optimal number of components (the number that explain 95% of variance observed)

# transforming the training and testing data using PCA
X_train_pca = pca.fit_transform(X_train) 
X_test_pca = pca.transform(X_test)

n_components_kept = pca.n_components_ 
print(n_components_kept) # prints the number of components kept = explain 95% of variance
# goes from 7987 components to 1385 (1/7 of the size)

1385


# 7. Hyperparameter Tuning with PCA

In [ ]:
# since PCA changes the dimension the data is in, need to re-perform hyperparameter tuning for dimension-reduced data

knn = KNeighborsClassifier()

bayes_search_pca = BayesSearchCV(estimator = knn,
                           search_spaces=search_space,
                           scoring = "f1_macro",
                           n_iter=50, # the number of hyperparameter combinations to try
                           cv = 5,
                           n_jobs=-1, 
                           random_state=42,
                           verbose=1 
)

bayes_search_pca.fit(X_train_pca, y_train_enc) # running grid search on PCA data

Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits


,estimator,KNeighborsClassifier()
,search_spaces,"{'metric': Categorical(c...), prior=None), 'n_neighbors': Integer(low=1...m='normalize'), 'weights': Categorical(c...), prior=None)}"
,optimizer_kwargs,None
,n_iter,2
,scoring,'f1_macro'
,fit_params,None
,n_jobs,-1
,n_points,1
,iid,'deprecated'
,refit,True
,cv,5


In [14]:
bayes_search_pca.best_params_

OrderedDict([('metric', 'euclidean'),
             ('n_neighbors', 71),
             ('weights', 'uniform')])

# Next steps
- Run bayes search with 50 iterations (or change this is if more/less makes sense)
- Analyze the hyperparamter tuning - plot of k vs F1
- Fit non-PCA and PCA data with best model, compare performance metrics, generate model predictions and analyze
    - Does PCA data perform better? Why is this/why does this make sense?
- May also want to add more performance metrics for hyperparameter tuning (accuracy, precision, etc)
    - I just initialized with F1 score to have something to look at, but based on the nature of the task determine which metrics make sense
- Interpret results: plots by genre based on the percentage of correct predictions, research into literature as to why some genres perform better than others
    - Get the performance metrics by genre